# 02 — Trích xuất đặc trưng (Feature Engineering)

Tạo cột Vùng miền, gom nhóm dân tộc, bóc tách & geocode địa chỉ, xử lý mã ICD thành các nhóm bệnh. **Đầu vào**: `data/processed/01_cleaned.pkl`. **Kết quả**: `data/processed/model_df.{pkl,csv}` cho EDA & Modeling.

> ⚠️ Cell geocoding gọi API Nominatim (~1 req/giây) nên có thể mất ~20 phút.

### Import & nạp dữ liệu đã làm sạch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import config

In [ ]:
df = pd.read_pickle(config.CLEANED_PKL)
df.head()

- Tạo một cột Miền địa lý để xem tỉnh nào thuộc miền nào

In [40]:
def region(provinces):
  regions_dict = {
    "Đồng bằng sông Hồng": ["HÀ NỘI", "HẢI PHÒNG", "HẢI DƯƠNG", "HƯNG YÊN", "HÀ NAM", "NAM ĐỊNH", "THÁI BÌNH", "NINH BÌNH", "VĨNH PHÚC", "BẮC NINH"],
    "Trung du và miền núi phía Bắc": ["LÀO CAI", "ĐIỆN BIÊN", "LAI CHÂU", "SƠN LA", "YÊN BÁI", "HÒA BÌNH", "CAO BẰNG", "BẮC KẠN",
                                      "HÀ GIANG", "TUYÊN QUANG", "THÁI NGUYÊN", "LẠNG SƠN", "BẮC GIANG", "PHÚ THỌ", "QUẢNG NINH"],
    "Bắc Trung Bộ": ["THANH HÓA", "NGHỆ AN", "HÀ TĨNH", "QUẢNG BÌNH", "QUẢNG TRỊ", "THỪA THIÊN - HUẾ"],
    "Nam Trung Bộ": ["ĐÀ NẴNG", "QUẢNG NAM", "QUẢNG NGÃI", "BÌNH ĐỊNH", "PHÚ YÊN", "KHÁNH HÒA", "NINH THUẬN", "BÌNH THUẬN"],
    "Tây Nguyên": ["KON TUM", "GIA LAI", "ĐẮK LẮK", "ĐẮK NÔNG", "LÂM ĐỒNG"],
    "Đông Nam Bộ": ["TP. HỒ CHÍ MINH", "BÌNH DƯƠNG", "ĐỒNG NAI", "BÀ RỊA - VŨNG TÀU", "TÂY NINH", "BÌNH PHƯỚC"],
    "Đồng bằng sông Cửu Long": ["CẦN THƠ", "LONG AN", "TIỀN GIANG", "BẾN TRE", "VĨNH LONG", "ĐỒNG THÁP", "AN GIANG", "KIÊN GIANG", "HẬU GIANG", "SÓC TRĂNG", "BẠC LIÊU", "CÀ MAU", "TRÀ VINH"]
  }
  regions = []
  for province in provinces:
    for region, province_list in regions_dict.items():
      if province in province_list:
        regions.append(region)
  return regions

df['Vùng miền'] = region(df['TENTINHTHANH'])

**Đếm số lượt khám theo từng vùng miền** để biết bệnh nhân chủ yếu đến từ đâu.

In [41]:
df['Vùng miền'].value_counts()

Vùng miền
Nam Trung Bộ                     64184
Tây Nguyên                        3298
Đông Nam Bộ                        223
Đồng bằng sông Hồng                144
Bắc Trung Bộ                       114
Đồng bằng sông Cửu Long             50
Trung du và miền núi phía Bắc       23
Name: count, dtype: int64

- Bây giờ đến bước xử lí cột DANTOC, có tổng cộng 27 dân tộc (tính cả người nước ngoài)

In [42]:
ethnicity = df['DANTOC'].unique()
print(ethnicity)
print(len(ethnicity))

<ArrowStringArray>
[      'Kinh',      'Ba na',       'Chăm',        'Lào',       'Ê đê',
     'M nông',      'La hù', 'Nước ngoài',      'Lô lô',       'H rê',
    'Gia rai',      'La ha',     'La chí',       'Nùng',    'X tiêng',
         'Lự',        'Tày',     'Hà nhì',        'Thổ',     'H mông',
      'Mường',       'Chứt',       'K tu',       'Thái',    'Xơ đăng',
        'Hoa',        'Dao']
Length: 27, dtype: str
27


In [43]:
df['DANTOC'] = list(map(lambda x: x if (x == 'Kinh' or x == 'Nước ngoài') else 'Dân tộc thiểu số',df['DANTOC']))

**Xem lại dữ liệu** sau khi đã gom nhóm dân tộc.

In [44]:
df

,Tuổi,DANTOC,TENPXA,TENQUANHUYEN,TENTINHTHANH,MAICD,CHANDOAN,NGAYVAO,NGAYRA,TONGCP,BHYT_TT,Thời gian điều trị,Vùng miền
0,28,Kinh,PHƯỜNG NHƠN BÌNH,THÀNH PHỐ QUI NHƠN,BÌNH ĐỊNH,S01.1;,Vết thương hở của mi mắt và vùng quanh mắt;,2016-01-01 00:55:00,2016-01-01 01:04:00,1.500000e+04,0.000000e+00,0.006250,Nam Trung Bộ
1,18,Kinh,PHƯỜNG NHƠN PHÚ,THÀNH PHỐ QUI NHƠN,BÌNH ĐỊNH,I20;,Cơn đau thắt ngực;,2016-01-01 01:37:00,2016-01-01 03:23:00,8.334650e+04,8.334650e+04,0.073611,Nam Trung Bộ
2,36,Kinh,XÃ NHƠN LỘC,THỊ XÃ AN NHƠN,BÌNH ĐỊNH,J68.2;P71.0;,Viêm hô hấp trên;Hạ calci máu;,2016-01-01 03:31:00,2016-01-01 05:00:00,1.599990e+02,0.000000e+00,0.061806,Nam Trung Bộ
3,5,Kinh,PHƯỜNG TRẦN QUANG DIỆU,THÀNH PHỐ QUI NHƠN,BÌNH ĐỊNH,A91.A;,Sốt xuất huyết Dengue;,2015-12-29 20:25:00,2016-01-01 07:00:00,1.129380e+05,1.129380e+05,2.440972,Nam Trung Bộ
4,5,Kinh,PHƯỜNG NHƠN PHÚ,THÀNH PHỐ QUI NHƠN,BÌNH ĐỊNH,A91;J00;,Sốt xuất huyết Dengue;Viêm Họng Cấp;,2015-12-25 08:44:00,2016-01-01 07:00:00,2.623950e+05,2.623950e+05,6.927778,Nam Trung Bộ
...,...,...,...,...,...,...,...,...,...,...,...,...,...
68757,73,Kinh,PHƯỜNG TRẦN QUANG DIỆU,THÀNH PHỐ QUI NHƠN,BÌNH ĐỊNH,Y84.1;D63.0;,Chạy thận nhân tạo;Thiếu máu trong bệnh suy th...,2016-12-01 08:11:00,2016-12-31 23:00:00,1.102239e+07,1.102239e+07,30.617361,Nam Trung Bộ
68758,24,Dân tộc thiểu số,PHƯỜNG BÌNH ĐỊNH,THỊ XÃ AN NHƠN,BÌNH ĐỊNH,Y84.1;,Chạy thận nhân tạo;,2016-11-19 06:53:00,2016-12-31 23:00:00,1.359227e+07,1.359227e+07,42.671528,Nam Trung Bộ
68759,42,Kinh,XÃ NHƠN PHÚC,THỊ XÃ AN NHƠN,BÌNH ĐỊNH,Y84.1;,Chạy thận nhân tạo;,2016-11-19 06:52:00,2016-12-31 23:00:00,1.021012e+07,1.021012e+07,42.672222,Nam Trung Bộ
68760,40,Kinh,XÃ MỸ THỌ,HUYỆN PHÙ MỸ,BÌNH ĐỊNH,Y84.1;J18;,Chạy thận nhân tạo;Viêm phổi;,2016-11-10 06:56:00,2016-12-31 23:00:00,1.734061e+07,1.734061e+07,51.669444,Nam Trung Bộ


**Khảo sát tiền tố tên Phường/Xã**: tách từ đầu tiên (PHƯỜNG, XÃ, THỊ TRẤN…) và đếm tần suất để biết cách bóc tách tên thuần.

In [45]:
df['TENPXA'].str.split(' ').str[0].value_counts()

TENPXA
XÃ        38166
PHƯỜNG    23976
THỊ        4586
PHƯỚC       402
TÂN         180
TT          163
KHÔNG        86
CƯ           84
ĐAK          76
HÀ           63
P.XUÂN       58
TÂY          50
HÒA          43
HOÀ          36
DĂK          14
BÌNH         10
HOÀNG         9
NAM           7
XÃ           4
PHÚ           4
ĐĂK           4
ĐẮK           3
HỒNG          2
H             1
SƠN           1
TRẢNG         1
NHƠN          1
LONG          1
XUÂN          1
ĐIỆN          1
XẢ            1
K             1
SÔNG          1
Name: count, dtype: int64

**Bóc tách tên Phường/Xã thuần** từ `TENPXA` (làm trên bản sao `df1`): bỏ tiền tố hành chính — PHƯỜNG/XÃ/TT/XẢ bỏ 1 từ; THỊ TRẤN bỏ 2 từ; xử lý riêng trường hợp `P.XUÂN`.

In [46]:
df1 = df.copy()
df1['Phường/Xã'] = list(map(lambda x: ' '.join(x.split(' ')[1:]) if (x.split(' ')[0] in ['PHƯỜNG','XÃ','TT','XẢ']) else
                                      ' '.join(x.split(' ')[2:]) if (x.split(' ')[0] == 'THỊ') else
                                      ' '.join(['XUÂN'] + x.split(' ')[1:]) if (x.split(' ')[0] == 'P.XUÂN') else x,df1['TENPXA']))

**Xử lý phường đánh số**: với tên phường chỉ gồm chữ số (vd '1', '2'), thêm tiền tố `'PHƯỜNG '` để tên có nghĩa (PHƯỜNG 1, PHƯỜNG 2…).

In [47]:
#df1[df1['Phường/Xã'].str.isdigit()]['TENPXA'].str.split(" ").str[0].value_counts()
df1['Phường/Xã'] = list(map(lambda x: 'PHƯỜNG ' + x if x.isdigit() else x,df1['Phường/Xã']))

**Đếm số Phường/Xã** duy nhất sau khi chuẩn hóa, để kiểm tra mức độ rút gọn.

In [48]:
df1['Phường/Xã'].nunique()

1054

**Khảo sát tiền tố tên Quận/Huyện** (HUYỆN, TP, THÀNH PHỐ, THỊ XÃ…) tương tự bước xử lý Phường/Xã.

In [49]:
df1['TENQUANHUYEN'].str.split(' ').str[0].value_counts()

TENQUANHUYEN
HUYỆN    37134
THÀNH    22220
THỊ       8393
QUẬN       237
TP          28
TX           9
HOÀNG        9
LỘC          2
PHÚ          2
BUÔN         1
CHÂU         1
Name: count, dtype: int64

**Bóc tách tên Huyện/TP thuần** từ `TENQUANHUYEN`: bỏ tiền tố HUYỆN/TP/TX (1 từ) hoặc THÀNH PHỐ/THỊ XÃ (2 từ).

In [50]:
df1['Huyện/TP'] = list(map(lambda x: ' '.join(x.split(' ')[1:]) if (x.split(' ')[0] in ['HUYỆN','TP','TX']) else
                                     ' '.join(x.split(' ')[2:]) if (x.split(' ')[0] in ['THÀNH','THỊ']) else x,df1['TENQUANHUYEN']))

**Xem lại dữ liệu** với hai cột địa danh đã chuẩn hóa (`Phường/Xã`, `Huyện/TP`).

In [51]:
df1

,Tuổi,DANTOC,TENPXA,TENQUANHUYEN,TENTINHTHANH,MAICD,CHANDOAN,NGAYVAO,NGAYRA,TONGCP,BHYT_TT,Thời gian điều trị,Vùng miền,Phường/Xã,Huyện/TP
0,28,Kinh,PHƯỜNG NHƠN BÌNH,THÀNH PHỐ QUI NHƠN,BÌNH ĐỊNH,S01.1;,Vết thương hở của mi mắt và vùng quanh mắt;,2016-01-01 00:55:00,2016-01-01 01:04:00,1.500000e+04,0.000000e+00,0.006250,Nam Trung Bộ,NHƠN BÌNH,QUI NHƠN
1,18,Kinh,PHƯỜNG NHƠN PHÚ,THÀNH PHỐ QUI NHƠN,BÌNH ĐỊNH,I20;,Cơn đau thắt ngực;,2016-01-01 01:37:00,2016-01-01 03:23:00,8.334650e+04,8.334650e+04,0.073611,Nam Trung Bộ,NHƠN PHÚ,QUI NHƠN
2,36,Kinh,XÃ NHƠN LỘC,THỊ XÃ AN NHƠN,BÌNH ĐỊNH,J68.2;P71.0;,Viêm hô hấp trên;Hạ calci máu;,2016-01-01 03:31:00,2016-01-01 05:00:00,1.599990e+02,0.000000e+00,0.061806,Nam Trung Bộ,NHƠN LỘC,AN NHƠN
3,5,Kinh,PHƯỜNG TRẦN QUANG DIỆU,THÀNH PHỐ QUI NHƠN,BÌNH ĐỊNH,A91.A;,Sốt xuất huyết Dengue;,2015-12-29 20:25:00,2016-01-01 07:00:00,1.129380e+05,1.129380e+05,2.440972,Nam Trung Bộ,TRẦN QUANG DIỆU,QUI NHƠN
4,5,Kinh,PHƯỜNG NHƠN PHÚ,THÀNH PHỐ QUI NHƠN,BÌNH ĐỊNH,A91;J00;,Sốt xuất huyết Dengue;Viêm Họng Cấp;,2015-12-25 08:44:00,2016-01-01 07:00:00,2.623950e+05,2.623950e+05,6.927778,Nam Trung Bộ,NHƠN PHÚ,QUI NHƠN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68757,73,Kinh,PHƯỜNG TRẦN QUANG DIỆU,THÀNH PHỐ QUI NHƠN,BÌNH ĐỊNH,Y84.1;D63.0;,Chạy thận nhân tạo;Thiếu máu trong bệnh suy th...,2016-12-01 08:11:00,2016-12-31 23:00:00,1.102239e+07,1.102239e+07,30.617361,Nam Trung Bộ,TRẦN QUANG DIỆU,QUI NHƠN
68758,24,Dân tộc thiểu số,PHƯỜNG BÌNH ĐỊNH,THỊ XÃ AN NHƠN,BÌNH ĐỊNH,Y84.1;,Chạy thận nhân tạo;,2016-11-19 06:53:00,2016-12-31 23:00:00,1.359227e+07,1.359227e+07,42.671528,Nam Trung Bộ,BÌNH ĐỊNH,AN NHƠN
68759,42,Kinh,XÃ NHƠN PHÚC,THỊ XÃ AN NHƠN,BÌNH ĐỊNH,Y84.1;,Chạy thận nhân tạo;,2016-11-19 06:52:00,2016-12-31 23:00:00,1.021012e+07,1.021012e+07,42.672222,Nam Trung Bộ,NHƠN PHÚC,AN NHƠN
68760,40,Kinh,XÃ MỸ THỌ,HUYỆN PHÙ MỸ,BÌNH ĐỊNH,Y84.1;J18;,Chạy thận nhân tạo;Viêm phổi;,2016-11-10 06:56:00,2016-12-31 23:00:00,1.734061e+07,1.734061e+07,51.669444,Nam Trung Bộ,MỸ THỌ,PHÙ MỸ


**Tạo bảng địa chỉ duy nhất** `df2`: lấy các tổ hợp (Phường/Xã, Huyện/TP, Tỉnh) không trùng và ghép thành chuỗi `'Địa chỉ'` đầy đủ (bỏ phần Phường/Xã nếu là `KHÔNG XÁC ĐỊNH`). Đây là bước chuẩn bị để **lấy tọa độ (geocode) và join với `address_df_full.csv`** ở phần tiếp theo.

In [52]:
df2 = df1[['Phường/Xã','Huyện/TP','TENTINHTHANH']].drop_duplicates()
df2['Địa chỉ'] = list(map(lambda x,y,z: x + ',' + y + ',' + z if (x != 'KHÔNG XÁC ĐỊNH') else
                                   y + ',' + z, df2['Phường/Xã'], df2['Huyện/TP'], df2['TENTINHTHANH']))
df2.reset_index(drop = True, inplace = True)
df2

,Phường/Xã,Huyện/TP,TENTINHTHANH,Địa chỉ
0,NHƠN BÌNH,QUI NHƠN,BÌNH ĐỊNH,"NHƠN BÌNH,QUI NHƠN,BÌNH ĐỊNH"
1,NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH,"NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH"
2,NHƠN LỘC,AN NHƠN,BÌNH ĐỊNH,"NHƠN LỘC,AN NHƠN,BÌNH ĐỊNH"
3,TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH,"TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH"
4,DIÊU TRÌ,TUY PHƯỚC,BÌNH ĐỊNH,"DIÊU TRÌ,TUY PHƯỚC,BÌNH ĐỊNH"
...,...,...,...,...
1252,ĐÔNG THẠNH,CẦN GIUỘC,LONG AN,"ĐÔNG THẠNH,CẦN GIUỘC,LONG AN"
1253,THANH TRÌ,THANH TRÌ,HÀ NỘI,"THANH TRÌ,THANH TRÌ,HÀ NỘI"
1254,LỘC THỦY,PHÚ LỘC,THỪA THIÊN - HUẾ,"LỘC THỦY,PHÚ LỘC,THỪA THIÊN - HUẾ"
1255,VĨNH THANH,RẠCH GIÁ,KIÊN GIANG,"VĨNH THANH,RẠCH GIÁ,KIÊN GIANG"


**Lấy tọa độ trực tiếp qua API geocoding (thay vì đọc `address_df_full.csv` có sẵn).**

Dùng dịch vụ **Nominatim (OpenStreetMap)** qua thư viện `geopy` — miễn phí, không cần API key. Với mỗi địa chỉ duy nhất trong `df2`, ta truy vấn tọa độ theo 2 mức:

- **level 1**: tìm thấy theo địa chỉ đầy đủ (Phường/Xã, Huyện/TP, Tỉnh).
- **level 2**: không thấy chi tiết → lùi dần về cấp Huyện rồi cấp Tỉnh (tọa độ này sẽ được *jitter* thêm nhiễu ở cell phía sau).
- Không tìm được → để trống (`NaN`), sẽ bị `dropna` loại bỏ sau.

Kết quả là DataFrame `coordinate` có đúng các cột như file cũ (`Địa chỉ mới, Tọa độ, level, kinh độ, vĩ độ`) nên các cell phía sau chạy y nguyên.

> ⚠️ Nominatim giới hạn ~1 request/giây. Với ~1.300 địa chỉ, cell này chạy mất **~20–25 phút**. Có thể bỏ comment dòng `to_csv` cuối để lưu lại, lần sau khỏi chạy lại.

In [53]:
# Cài đặt thư viện geocoding (chỉ cần chạy 1 lần)
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Nominatim (OpenStreetMap): miễn phí, KHÔNG cần API key.
# Bắt buộc đặt user_agent riêng và tuân thủ giới hạn 1 request/giây.
geolocator = Nominatim(user_agent=config.GEOCODER_USER_AGENT)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=config.GEOCODER_MIN_DELAY,
                      max_retries=2, swallow_exceptions=True)

# Cache để không truy vấn lặp lại cùng một chuỗi (vd các fallback cấp tỉnh)
_geo_cache = {}
def _query(q):
    if q not in _geo_cache:
        loc = geocode(q + ', Việt Nam', country_codes='vn')
        _geo_cache[q] = (loc.latitude, loc.longitude) if loc else None
    return _geo_cache[q]

def geocode_address(addr):
    """Trả về (vĩ độ, kinh độ, level). level=1 nếu khớp đầy đủ, =2 nếu phải fallback."""
    parts = [p.strip().title() for p in addr.split(',')]
    # Mức 1: địa chỉ đầy đủ
    res = _query(', '.join(parts))
    if res:
        return res[0], res[1], 1
    # Mức 2: lùi dần - bỏ phần đầu (Phường/Xã), rồi đến khi chỉ còn Tỉnh
    for i in range(1, len(parts)):
        res = _query(', '.join(parts[i:]))
        if res:
            return res[0], res[1], 2
    return None, None, None

ModuleNotFoundError: No module named 'geopy'

In [ ]:
# Geocode toàn bộ địa chỉ duy nhất trong df2 (giữ nguyên thứ tự để concat ăn khớp)
records = []
for n, addr in enumerate(df2['Địa chỉ'], start=1):
    lat, lon, level = geocode_address(addr)
    records.append({
        'Địa chỉ mới': addr,
        'Tọa độ': (lat, lon) if lat is not None else None,
        'level': level,
        'kinh độ': lon,   # kinh độ = longitude (~109)
        'vĩ độ': lat,     # vĩ độ  = latitude  (~13-23)
    })
    if n % 100 == 0:
        print(f'  ...đã geocode {n}/{len(df2)} địa chỉ')

coordinate = pd.DataFrame(records)
# Lưu lại để lần sau không phải chạy lại (tùy chọn):
# coordinate.to_csv('address_df_full.csv', index=False)
coordinate

In [ ]:
concat_df = pd.concat([df2,coordinate], axis = 1)
concat_df

,Phường/Xã,Huyện/TP,TENTINHTHANH,Địa chỉ,Địa chỉ mới,Tọa độ,level,kinh độ,vĩ độ
0,NHƠN BÌNH,QUI NHƠN,BÌNH ĐỊNH,"NHƠN BÌNH,QUI NHƠN,BÌNH ĐỊNH","NHƠN BÌNH,QUI NHƠN,BÌNH ĐỊNH","(13.7930532, 109.1916057)",1,109.191606,13.793053
1,NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH,"NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH","NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH","(13.7945583, 109.1813114)",1,109.181311,13.794558
2,NHƠN LỘC,AN NHƠN,BÌNH ĐỊNH,"NHƠN LỘC,AN NHƠN,BÌNH ĐỊNH","NHƠN LỘC,AN NHƠN,BÌNH ĐỊNH","(13.8789863, 109.0686007)",1,109.068601,13.878986
3,TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH,"TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH","TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH","(13.786352, 109.1482711)",1,109.148271,13.786352
4,DIÊU TRÌ,TUY PHƯỚC,BÌNH ĐỊNH,"DIÊU TRÌ,TUY PHƯỚC,BÌNH ĐỊNH","DIÊU TRÌ,TUY PHƯỚC,BÌNH ĐỊNH","(13.8071597, 109.1437149)",1,109.143715,13.807160
...,...,...,...,...,...,...,...,...,...
1252,ĐÔNG THẠNH,CẦN GIUỘC,LONG AN,"ĐÔNG THẠNH,CẦN GIUỘC,LONG AN","ĐÔNG THẠNH,CẦN GIUỘC,LONG AN","(10.5295518, 106.6724105)",1,106.672410,10.529552
1253,THANH TRÌ,THANH TRÌ,HÀ NỘI,"THANH TRÌ,THANH TRÌ,HÀ NỘI","THANH TRÌ,THANH TRÌ,HÀ NỘI","(20.94805855, 105.84967889687385)",1,105.849679,20.948059
1254,LỘC THỦY,PHÚ LỘC,THỪA THIÊN - HUẾ,"LỘC THỦY,PHÚ LỘC,THỪA THIÊN - HUẾ","LỘC THỦY,PHÚ LỘC,THỪA THIÊN - HUẾ","(16.2706984, 107.9277907)",1,107.927791,16.270698
1255,VĨNH THANH,RẠCH GIÁ,KIÊN GIANG,"VĨNH THANH,RẠCH GIÁ,KIÊN GIANG","VĨNH THANH,RẠCH GIÁ,KIÊN GIANG","(10.0153837, 105.0831127)",1,105.083113,10.015384


In [ ]:
df1['Địa chỉ'] = list(map(lambda x,y,z: x + ',' + y + ',' + z if (x != 'KHÔNG XÁC ĐỊNH') else
                                        y + ',' + z, df1['Phường/Xã'], df1['Huyện/TP'], df1['TENTINHTHANH']))
df1 = df1.drop(columns = ['TENPXA','TENQUANHUYEN'])
df1

,Tuổi,DANTOC,TENTINHTHANH,MAICD,CHANDOAN,NGAYVAO,NGAYRA,TONGCP,BHYT_TT,Thời gian điều trị,Vùng miền,Phường/Xã,Huyện/TP,Địa chỉ
0,28,Kinh,BÌNH ĐỊNH,S01.1;,Vết thương hở của mi mắt và vùng quanh mắt;,2016-01-01 00:55:00,2016-01-01 01:04:00,1.500000e+04,0.000000e+00,0.006250,Nam Trung Bộ,NHƠN BÌNH,QUI NHƠN,"NHƠN BÌNH,QUI NHƠN,BÌNH ĐỊNH"
1,18,Kinh,BÌNH ĐỊNH,I20;,Cơn đau thắt ngực;,2016-01-01 01:37:00,2016-01-01 03:23:00,8.334650e+04,8.334650e+04,0.073611,Nam Trung Bộ,NHƠN PHÚ,QUI NHƠN,"NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH"
2,36,Kinh,BÌNH ĐỊNH,J68.2;P71.0;,Viêm hô hấp trên;Hạ calci máu;,2016-01-01 03:31:00,2016-01-01 05:00:00,1.599990e+02,0.000000e+00,0.061806,Nam Trung Bộ,NHƠN LỘC,AN NHƠN,"NHƠN LỘC,AN NHƠN,BÌNH ĐỊNH"
3,5,Kinh,BÌNH ĐỊNH,A91.A;,Sốt xuất huyết Dengue;,2015-12-29 20:25:00,2016-01-01 07:00:00,1.129380e+05,1.129380e+05,2.440972,Nam Trung Bộ,TRẦN QUANG DIỆU,QUI NHƠN,"TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH"
4,5,Kinh,BÌNH ĐỊNH,A91;J00;,Sốt xuất huyết Dengue;Viêm Họng Cấp;,2015-12-25 08:44:00,2016-01-01 07:00:00,2.623950e+05,2.623950e+05,6.927778,Nam Trung Bộ,NHƠN PHÚ,QUI NHƠN,"NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68757,73,Kinh,BÌNH ĐỊNH,Y84.1;D63.0;,Chạy thận nhân tạo;Thiếu máu trong bệnh suy th...,2016-12-01 08:11:00,2016-12-31 23:00:00,1.102239e+07,1.102239e+07,30.617361,Nam Trung Bộ,TRẦN QUANG DIỆU,QUI NHƠN,"TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH"
68758,24,Dân tộc thiểu số,BÌNH ĐỊNH,Y84.1;,Chạy thận nhân tạo;,2016-11-19 06:53:00,2016-12-31 23:00:00,1.359227e+07,1.359227e+07,42.671528,Nam Trung Bộ,BÌNH ĐỊNH,AN NHƠN,"BÌNH ĐỊNH,AN NHƠN,BÌNH ĐỊNH"
68759,42,Kinh,BÌNH ĐỊNH,Y84.1;,Chạy thận nhân tạo;,2016-11-19 06:52:00,2016-12-31 23:00:00,1.021012e+07,1.021012e+07,42.672222,Nam Trung Bộ,NHƠN PHÚC,AN NHƠN,"NHƠN PHÚC,AN NHƠN,BÌNH ĐỊNH"
68760,40,Kinh,BÌNH ĐỊNH,Y84.1;J18;,Chạy thận nhân tạo;Viêm phổi;,2016-11-10 06:56:00,2016-12-31 23:00:00,1.734061e+07,1.734061e+07,51.669444,Nam Trung Bộ,MỸ THỌ,PHÙ MỸ,"MỸ THỌ,PHÙ MỸ,BÌNH ĐỊNH"


In [ ]:
df1['Kinh độ'] = [concat_df[concat_df['Địa chỉ'] == x]['kinh độ'].iloc[0] for x in df1['Địa chỉ']]
df1['Vĩ độ'] = [concat_df[concat_df['Địa chỉ'] == x]['vĩ độ'].iloc[0] for x in df1['Địa chỉ']]
df1['level'] = [concat_df[concat_df['Địa chỉ'] == x]['level'].iloc[0] for x in df1['Địa chỉ']]

In [ ]:
# Đặt seed để bước jitter (cộng nhiễu ngẫu nhiên) tái lập được giữa các lần chạy
np.random.seed(config.RANDOM_SEED)
df1['Kinh độ'] = list(map(lambda x,y: x if (y == 1) else x + np.random.uniform(0.001,0.002),df1['Kinh độ'],df1['level']))
df1['Vĩ độ'] = list(map(lambda x,y: x if (y == 1) else x + np.random.uniform(0.001,0.002),df1['Vĩ độ'],df1['level']))

In [ ]:
df1 = df1.dropna(subset = ['Kinh độ'])
df1

,Tuổi,DANTOC,TENTINHTHANH,MAICD,CHANDOAN,NGAYVAO,NGAYRA,TONGCP,BHYT_TT,Thời gian điều trị,Vùng miền,Phường/Xã,Huyện/TP,Địa chỉ,Kinh độ,Vĩ độ,level
0,28,Kinh,BÌNH ĐỊNH,S01.1;,Vết thương hở của mi mắt và vùng quanh mắt;,2016-01-01 00:55:00,2016-01-01 01:04:00,1.500000e+04,0.000000e+00,0.006250,Nam Trung Bộ,NHƠN BÌNH,QUI NHƠN,"NHƠN BÌNH,QUI NHƠN,BÌNH ĐỊNH",109.191606,13.793053,1
1,18,Kinh,BÌNH ĐỊNH,I20;,Cơn đau thắt ngực;,2016-01-01 01:37:00,2016-01-01 03:23:00,8.334650e+04,8.334650e+04,0.073611,Nam Trung Bộ,NHƠN PHÚ,QUI NHƠN,"NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH",109.181311,13.794558,1
2,36,Kinh,BÌNH ĐỊNH,J68.2;P71.0;,Viêm hô hấp trên;Hạ calci máu;,2016-01-01 03:31:00,2016-01-01 05:00:00,1.599990e+02,0.000000e+00,0.061806,Nam Trung Bộ,NHƠN LỘC,AN NHƠN,"NHƠN LỘC,AN NHƠN,BÌNH ĐỊNH",109.068601,13.878986,1
3,5,Kinh,BÌNH ĐỊNH,A91.A;,Sốt xuất huyết Dengue;,2015-12-29 20:25:00,2016-01-01 07:00:00,1.129380e+05,1.129380e+05,2.440972,Nam Trung Bộ,TRẦN QUANG DIỆU,QUI NHƠN,"TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH",109.148271,13.786352,1
4,5,Kinh,BÌNH ĐỊNH,A91;J00;,Sốt xuất huyết Dengue;Viêm Họng Cấp;,2015-12-25 08:44:00,2016-01-01 07:00:00,2.623950e+05,2.623950e+05,6.927778,Nam Trung Bộ,NHƠN PHÚ,QUI NHƠN,"NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH",109.181311,13.794558,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68757,73,Kinh,BÌNH ĐỊNH,Y84.1;D63.0;,Chạy thận nhân tạo;Thiếu máu trong bệnh suy th...,2016-12-01 08:11:00,2016-12-31 23:00:00,1.102239e+07,1.102239e+07,30.617361,Nam Trung Bộ,TRẦN QUANG DIỆU,QUI NHƠN,"TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH",109.148271,13.786352,1
68758,24,Dân tộc thiểu số,BÌNH ĐỊNH,Y84.1;,Chạy thận nhân tạo;,2016-11-19 06:53:00,2016-12-31 23:00:00,1.359227e+07,1.359227e+07,42.671528,Nam Trung Bộ,BÌNH ĐỊNH,AN NHƠN,"BÌNH ĐỊNH,AN NHƠN,BÌNH ĐỊNH",109.101382,13.887191,1
68759,42,Kinh,BÌNH ĐỊNH,Y84.1;,Chạy thận nhân tạo;,2016-11-19 06:52:00,2016-12-31 23:00:00,1.021012e+07,1.021012e+07,42.672222,Nam Trung Bộ,NHƠN PHÚC,AN NHƠN,"NHƠN PHÚC,AN NHƠN,BÌNH ĐỊNH",109.030093,13.906113,1
68760,40,Kinh,BÌNH ĐỊNH,Y84.1;J18;,Chạy thận nhân tạo;Viêm phổi;,2016-11-10 06:56:00,2016-12-31 23:00:00,1.734061e+07,1.734061e+07,51.669444,Nam Trung Bộ,MỸ THỌ,PHÙ MỸ,"MỸ THỌ,PHÙ MỸ,BÌNH ĐỊNH",109.148393,14.220117,1


In [ ]:
df1['MAICD'].str[-1].value_counts()

,count
MAICD,
;,67850
3,106
5,24
7,16
2,9
1,7
0,6
4,5
9,2


In [ ]:
df1['MAICD'] = list(map(lambda x: x if x[-1] == ';' else x + ';',df1['MAICD']))

In [ ]:
df1['Số lượng bệnh/hỗ trợ y tế'] = [len(x) + 1 if (len(x) == 0) else len(x) for x in df1['MAICD'].str.split(';').str[0:-1]]
df1

,Tuổi,DANTOC,TENTINHTHANH,MAICD,CHANDOAN,NGAYVAO,NGAYRA,TONGCP,BHYT_TT,Thời gian điều trị,Vùng miền,Phường/Xã,Huyện/TP,Địa chỉ,Kinh độ,Vĩ độ,level,Số lượng bệnh/hỗ trợ y tế
0,28,Kinh,BÌNH ĐỊNH,S01.1;,Vết thương hở của mi mắt và vùng quanh mắt;,2016-01-01 00:55:00,2016-01-01 01:04:00,1.500000e+04,0.000000e+00,0.006250,Nam Trung Bộ,NHƠN BÌNH,QUI NHƠN,"NHƠN BÌNH,QUI NHƠN,BÌNH ĐỊNH",109.191606,13.793053,1,1
1,18,Kinh,BÌNH ĐỊNH,I20;,Cơn đau thắt ngực;,2016-01-01 01:37:00,2016-01-01 03:23:00,8.334650e+04,8.334650e+04,0.073611,Nam Trung Bộ,NHƠN PHÚ,QUI NHƠN,"NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH",109.181311,13.794558,1,1
2,36,Kinh,BÌNH ĐỊNH,J68.2;P71.0;,Viêm hô hấp trên;Hạ calci máu;,2016-01-01 03:31:00,2016-01-01 05:00:00,1.599990e+02,0.000000e+00,0.061806,Nam Trung Bộ,NHƠN LỘC,AN NHƠN,"NHƠN LỘC,AN NHƠN,BÌNH ĐỊNH",109.068601,13.878986,1,2
3,5,Kinh,BÌNH ĐỊNH,A91.A;,Sốt xuất huyết Dengue;,2015-12-29 20:25:00,2016-01-01 07:00:00,1.129380e+05,1.129380e+05,2.440972,Nam Trung Bộ,TRẦN QUANG DIỆU,QUI NHƠN,"TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH",109.148271,13.786352,1,1
4,5,Kinh,BÌNH ĐỊNH,A91;J00;,Sốt xuất huyết Dengue;Viêm Họng Cấp;,2015-12-25 08:44:00,2016-01-01 07:00:00,2.623950e+05,2.623950e+05,6.927778,Nam Trung Bộ,NHƠN PHÚ,QUI NHƠN,"NHƠN PHÚ,QUI NHƠN,BÌNH ĐỊNH",109.181311,13.794558,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68757,73,Kinh,BÌNH ĐỊNH,Y84.1;D63.0;,Chạy thận nhân tạo;Thiếu máu trong bệnh suy th...,2016-12-01 08:11:00,2016-12-31 23:00:00,1.102239e+07,1.102239e+07,30.617361,Nam Trung Bộ,TRẦN QUANG DIỆU,QUI NHƠN,"TRẦN QUANG DIỆU,QUI NHƠN,BÌNH ĐỊNH",109.148271,13.786352,1,2
68758,24,Dân tộc thiểu số,BÌNH ĐỊNH,Y84.1;,Chạy thận nhân tạo;,2016-11-19 06:53:00,2016-12-31 23:00:00,1.359227e+07,1.359227e+07,42.671528,Nam Trung Bộ,BÌNH ĐỊNH,AN NHƠN,"BÌNH ĐỊNH,AN NHƠN,BÌNH ĐỊNH",109.101382,13.887191,1,1
68759,42,Kinh,BÌNH ĐỊNH,Y84.1;,Chạy thận nhân tạo;,2016-11-19 06:52:00,2016-12-31 23:00:00,1.021012e+07,1.021012e+07,42.672222,Nam Trung Bộ,NHƠN PHÚC,AN NHƠN,"NHƠN PHÚC,AN NHƠN,BÌNH ĐỊNH",109.030093,13.906113,1,1
68760,40,Kinh,BÌNH ĐỊNH,Y84.1;J18;,Chạy thận nhân tạo;Viêm phổi;,2016-11-10 06:56:00,2016-12-31 23:00:00,1.734061e+07,1.734061e+07,51.669444,Nam Trung Bộ,MỸ THỌ,PHÙ MỸ,"MỸ THỌ,PHÙ MỸ,BÌNH ĐỊNH",109.148393,14.220117,1,2


In [ ]:
def test():
  codes = df1['MAICD'].str.split(';').str[0:-1]
  codes_list = [x for code in codes for x in code]
  codes_list = list(set(codes_list))
  codes_list.sort()
  return codes_list

test()

['A00',
 'A01',
 'A01.0',
 'A02',
 'A02.0',
 'A02.1',
 'A02.8',
 'A03',
 'A03.9',
 'A04',
 'A04.0',
 'A04.5',
 'A04.8',
 'A04.9',
 'A05',
 'A05.1',
 'A06',
 'A06.0',
 'A06.1',
 'A06.2',
 'A06.3',
 'A06.4',
 'A06.5',
 'A06.6',
 'A08',
 'A08.0',
 'A08.3',
 'A08.5',
 'A09',
 'A09.0',
 'A09.9',
 'A15',
 'A15.0',
 'A15.1',
 'A15.2',
 'A15.3',
 'A15.4',
 'A15.6',
 'A16.0',
 'A16.1',
 'A16.2',
 'A16.4',
 'A17',
 'A17.0',
 'A18',
 'A18.0',
 'A18.3',
 'A19',
 'A20.3',
 'A20.7',
 'A21.0',
 'A23.1',
 'A24.3',
 'A25',
 'A26.7',
 'A28.1',
 'A28.8',
 'A31',
 'A31.8',
 'A32.0',
 'A32.1',
 'A32.9',
 'A33',
 'A35',
 'A37',
 'A37.9',
 'A39.0',
 'A39.1',
 'A40',
 'A40.9',
 'A41',
 'A41.0',
 'A41.2',
 'A41.4',
 'A41.8',
 'A41.9',
 'A42',
 'A48',
 'A48.3',
 'A48.4',
 'A48.8',
 'A49',
 'A49.0',
 'A49.8',
 'A49.9',
 'A54.2',
 'A54.5',
 'A55',
 'A57',
 'A58',
 'A68',
 'A68.0',
 'A68.9',
 'A69.0',
 'A74.0',
 'A75',
 'A75.2',
 'A75.9',
 'A77.2',
 'A78',
 'A79.0',
 'A79.8',
 'A80.3',
 'A81.1',
 'A82',
 'A82.9',


In [ ]:
!pip install roman
import roman

In [ ]:
def ICD(code):
  code = code[:3]
  code_range = [
      ('A00','B99'), ('C00','D49'), ('D50','D89'), ('E00','E89'), ('F00','F99'), ('G00','G99'), ('H00','H59'), ('H60','H95'),
      ('I00','I99'), ('J00','J99'), ('K00','K95'), ('L00','L99'), ('M00','M99'), ('N00','N99'), ('O00','O9A'), ('P00','P96'),
      ('Q00','Q99'), ('R00','R99'), ('S00','T98'), ('U00','U85'), ('V00','Y99'), ('Z00','Z99')
      ]
  for index in range(len(code_range)):
    if code >= code_range[index][0] and code <= code_range[index][1]:
      return roman.toRoman(index + 1)
  return None

In [ ]:
df1['MAICD'] = df1['MAICD'].str.split(';').str[0:-1]
df1['Danh mục'] = df1['MAICD'].apply(lambda x: [ICD(code) for code in x])
for i in range(1,23):
  df1['Số lượng danh mục bệnh %s'%(roman.toRoman(i))] = [x.count(roman.toRoman(i)) for x in df1['Danh mục']]

In [ ]:
df1

,Tuổi,DANTOC,TENTINHTHANH,MAICD,CHANDOAN,NGAYVAO,NGAYRA,TONGCP,BHYT_TT,Thời gian điều trị,...,Số lượng danh mục bệnh XIII,Số lượng danh mục bệnh XIV,Số lượng danh mục bệnh XV,Số lượng danh mục bệnh XVI,Số lượng danh mục bệnh XVII,Số lượng danh mục bệnh XVIII,Số lượng danh mục bệnh XIX,Số lượng danh mục bệnh XX,Số lượng danh mục bệnh XXI,Số lượng danh mục bệnh XXII
0,28,Kinh,BÌNH ĐỊNH,[S01.1],Vết thương hở của mi mắt và vùng quanh mắt;,2016-01-01 00:55:00,2016-01-01 01:04:00,1.500000e+04,0.000000e+00,0.006250,...,0,0,0,0,0,0,1,0,0,0
1,18,Kinh,BÌNH ĐỊNH,[I20],Cơn đau thắt ngực;,2016-01-01 01:37:00,2016-01-01 03:23:00,8.334650e+04,8.334650e+04,0.073611,...,0,0,0,0,0,0,0,0,0,0
2,36,Kinh,BÌNH ĐỊNH,"[J68.2, P71.0]",Viêm hô hấp trên;Hạ calci máu;,2016-01-01 03:31:00,2016-01-01 05:00:00,1.599990e+02,0.000000e+00,0.061806,...,0,0,0,1,0,0,0,0,0,0
3,5,Kinh,BÌNH ĐỊNH,[A91.A],Sốt xuất huyết Dengue;,2015-12-29 20:25:00,2016-01-01 07:00:00,1.129380e+05,1.129380e+05,2.440972,...,0,0,0,0,0,0,0,0,0,0
4,5,Kinh,BÌNH ĐỊNH,"[A91, J00]",Sốt xuất huyết Dengue;Viêm Họng Cấp;,2015-12-25 08:44:00,2016-01-01 07:00:00,2.623950e+05,2.623950e+05,6.927778,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68757,73,Kinh,BÌNH ĐỊNH,"[Y84.1, D63.0]",Chạy thận nhân tạo;Thiếu máu trong bệnh suy th...,2016-12-01 08:11:00,2016-12-31 23:00:00,1.102239e+07,1.102239e+07,30.617361,...,0,0,0,0,0,0,0,0,1,0
68758,24,Dân tộc thiểu số,BÌNH ĐỊNH,[Y84.1],Chạy thận nhân tạo;,2016-11-19 06:53:00,2016-12-31 23:00:00,1.359227e+07,1.359227e+07,42.671528,...,0,0,0,0,0,0,0,0,1,0
68759,42,Kinh,BÌNH ĐỊNH,[Y84.1],Chạy thận nhân tạo;,2016-11-19 06:52:00,2016-12-31 23:00:00,1.021012e+07,1.021012e+07,42.672222,...,0,0,0,0,0,0,0,0,1,0
68760,40,Kinh,BÌNH ĐỊNH,"[Y84.1, J18]",Chạy thận nhân tạo;Viêm phổi;,2016-11-10 06:56:00,2016-12-31 23:00:00,1.734061e+07,1.734061e+07,51.669444,...,0,0,0,0,0,0,0,0,1,0


In [ ]:
model_df = df1.copy().drop(columns = ['MAICD','Danh mục','TENTINHTHANH','CHANDOAN','Phường/Xã','Huyện/TP','NGAYRA','BHYT_TT','Địa chỉ','level','Danh mục'])
model_df['NGAYVAO'] = model_df['NGAYVAO'].dt.month
model_df = model_df.rename(columns = {'NGAYVAO': 'Tháng nhập viện'})
model_df = model_df[~((model_df['Vĩ độ'] > 20) & (model_df['Vùng miền'] == 'Nam Trung Bộ'))]
model_df

,Tuổi,DANTOC,Tháng nhập viện,TONGCP,Thời gian điều trị,Vùng miền,Kinh độ,Vĩ độ,Số lượng bệnh/hỗ trợ y tế,Số lượng danh mục bệnh I,...,Số lượng danh mục bệnh XIII,Số lượng danh mục bệnh XIV,Số lượng danh mục bệnh XV,Số lượng danh mục bệnh XVI,Số lượng danh mục bệnh XVII,Số lượng danh mục bệnh XVIII,Số lượng danh mục bệnh XIX,Số lượng danh mục bệnh XX,Số lượng danh mục bệnh XXI,Số lượng danh mục bệnh XXII
0,28,Kinh,1,1.500000e+04,0.006250,Nam Trung Bộ,109.191606,13.793053,1,0,...,0,0,0,0,0,0,1,0,0,0
1,18,Kinh,1,8.334650e+04,0.073611,Nam Trung Bộ,109.181311,13.794558,1,0,...,0,0,0,0,0,0,0,0,0,0
2,36,Kinh,1,1.599990e+02,0.061806,Nam Trung Bộ,109.068601,13.878986,2,0,...,0,0,0,1,0,0,0,0,0,0
3,5,Kinh,12,1.129380e+05,2.440972,Nam Trung Bộ,109.148271,13.786352,1,1,...,0,0,0,0,0,0,0,0,0,0
4,5,Kinh,12,2.623950e+05,6.927778,Nam Trung Bộ,109.181311,13.794558,2,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68757,73,Kinh,12,1.102239e+07,30.617361,Nam Trung Bộ,109.148271,13.786352,2,0,...,0,0,0,0,0,0,0,0,1,0
68758,24,Dân tộc thiểu số,11,1.359227e+07,42.671528,Nam Trung Bộ,109.101382,13.887191,1,0,...,0,0,0,0,0,0,0,0,1,0
68759,42,Kinh,11,1.021012e+07,42.672222,Nam Trung Bộ,109.030093,13.906113,1,0,...,0,0,0,0,0,0,0,0,1,0
68760,40,Kinh,11,1.734061e+07,51.669444,Nam Trung Bộ,109.148393,14.220117,2,0,...,0,0,0,0,0,0,0,0,1,0


### Lưu bảng dữ liệu cho mô hình
Lưu `model_df` ra cả `.pkl` (giữ kiểu dữ liệu) và `.csv` (dễ xem).

In [ ]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
model_df.to_pickle(config.MODEL_DF_PKL)
model_df.to_csv(config.MODEL_DF_CSV, index=False)
print('Đã lưu', model_df.shape, '->', config.MODEL_DF_PKL, '&', config.MODEL_DF_CSV)